# Chapter 8: sort 300 notes into five defect classes

Word counts weighted by rarity (TF-IDF) feed a classifier. 200 notes to learn from, 100 to test. Synthetic data only; no real brewery's numbers.

What to look at: precision and recall per class, then the safety rule.

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.pipeline import make_pipeline

c = pd.read_csv('../data/complaints.csv')
tr, te = c[c.split == 'train'], c[c.split == 'test']
print(c.defect_class.value_counts())

In [ ]:
model = make_pipeline(TfidfVectorizer(ngram_range=(1, 2)), LogisticRegression(max_iter=2000, C=3)).fit(tr.note, tr.defect_class)
pred = model.predict(te.note)
print(classification_report(te.defect_class, pred, zero_division=0))

In [ ]:
d = te.defect_class == 'diacetyl'
flag = pred == 'diacetyl'
print('Truly diacetyl:', d.sum(), 'flagged:', flag.sum(), 'right:', (d & flag).sum())
print(f'Recall {(d & flag).sum() / d.sum():.0%}, precision {(d & flag).sum() / flag.sum():.0%}')

## Rules around the model

Any note with a safety word goes to a person, whatever the model says.

In [ ]:
SAFETY = ['glass', 'metal', 'plastic', 'swallowed']
te = te.assign(model=pred, to_person=te.note.str.lower().str.contains('|'.join(SAFETY)))
print(te[te.to_person][['note', 'model']].head(10))